# OpenAI Function Calling (Modernized)

> **Note**: Updated to use the modern OpenAI Python SDK (`openai>=1.0`) with `client.chat.completions.create()`, `tools` parameter (replaces `functions`), and `tool_choice` (replaces `function_call`).

**Notes**:
- LLMs don't always produce the same results. The results you see in this notebook may differ from the results you see in the video.
- This notebook uses the modern `tools` API instead of the deprecated `functions` API.

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

client = OpenAI()
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o")

In [2]:
import json

# Example dummy function hard coded to return the same weather
# In production, this could be your backend API or an external API
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)

In [3]:
# define tools (replaces deprecated 'functions' parameter)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

In [4]:
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston?"
    }
]

In [5]:
# client.chat.completions.create (replaces openai.ChatCompletion.create)
response = client.chat.completions.create(
    model=llm_model,
    messages=messages,
    tools=tools,
)

> Note: The following result may differ slightly from the one shown by the instructor in the video lesson due to the model being updated.

In [6]:
print(response)

ChatCompletion(id='chatcmpl-DSaCotKz2pShxcSDkW3vjPEllUmnx', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XjBG0kyciA8A1knxktL8FBRg', function=Function(arguments='{"location":"Boston, MA"}', name='get_current_weather'), type='function')]))], created=1775704730, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_e2564de534', usage=CompletionUsage(completion_tokens=17, prompt_tokens=79, total_tokens=96, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [7]:
# Attribute access (replaces dict-style response["choices"][0]["message"])
response_message = response.choices[0].message

In [8]:
response_message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XjBG0kyciA8A1knxktL8FBRg', function=Function(arguments='{"location":"Boston, MA"}', name='get_current_weather'), type='function')])

In [9]:
response_message.content

In [10]:
# tool_calls replaces function_call
response_message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_XjBG0kyciA8A1knxktL8FBRg', function=Function(arguments='{"location":"Boston, MA"}', name='get_current_weather'), type='function')]

In [11]:
json.loads(response_message.tool_calls[0].function.arguments)

{'location': 'Boston, MA'}

In [12]:
args = json.loads(response_message.tool_calls[0].function.arguments)

In [13]:
get_current_weather(args)

'{"location": {"location": "Boston, MA"}, "temperature": "72", "unit": "fahrenheit", "forecast": ["sunny", "windy"]}'

* Pass a message that is not related to a function.

In [14]:
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]

In [15]:
response = client.chat.completions.create(
    model=llm_model,
    messages=messages,
    tools=tools,
)

In [16]:
print(response)

ChatCompletion(id='chatcmpl-DSaDn0u3pcSATLfFCDpjGT43EXGuZ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1775704791, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_e2564de534', usage=CompletionUsage(completion_tokens=10, prompt_tokens=74, total_tokens=84, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


* Pass additional parameters to force the model to use or not a function.

In [17]:
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
# tool_choice="auto" (replaces function_call="auto")
response = client.chat.completions.create(
    model=llm_model,
    messages=messages,
    tools=tools,
    tool_choice="auto",
)
print(response)

ChatCompletion(id='chatcmpl-DSaDsyDopNztbYU4xZvIgNPjRHS1Y', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1775704796, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_e2564de534', usage=CompletionUsage(completion_tokens=10, prompt_tokens=74, total_tokens=84, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


* Use mode 'none' for function call.

In [18]:
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
# tool_choice="none" (replaces function_call="none")
response = client.chat.completions.create(
    model=llm_model,
    messages=messages,
    tools=tools,
    tool_choice="none",
)
print(response)

ChatCompletion(id='chatcmpl-DSaDyXMuWMaCj7mrxSwrnqtz98nGF', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1775704802, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_e2564de534', usage=CompletionUsage(completion_tokens=9, prompt_tokens=75, total_tokens=84, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


* When the message should call a function and still uses mode 'none'.

In [19]:
messages = [
    {
        "role": "user",
        "content": "What's the weather in Boston?",
    }
]
# tool_choice="none" prevents tool use even when relevant
response = client.chat.completions.create(
    model=llm_model,
    messages=messages,
    tools=tools,
    tool_choice="none",
)
print(response)

ChatCompletion(id='chatcmpl-DSaEIIDqhZc8bzmGAfzvQLaDpv99m', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Would you like the weather information in Celsius or Fahrenheit?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1775704822, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_e2564de534', usage=CompletionUsage(completion_tokens=11, prompt_tokens=79, total_tokens=90, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


* Force calling a function.

In [20]:
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]
# Force calling a specific tool (replaces function_call={"name": ...})
response = client.chat.completions.create(
    model=llm_model,
    messages=messages,
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "get_current_weather"}},
)
print(response)

ChatCompletion(id='chatcmpl-DSaEUl9ouBc23bLMlrbe75Q5WAZG7', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_OMZD77fF8sqjGMB5K88mUB3v', function=Function(arguments='{"location":"Moscow"}', name='get_current_weather'), type='function')]))], created=1775704834, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_e2564de534', usage=CompletionUsage(completion_tokens=6, prompt_tokens=84, total_tokens=90, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


* Final notes.

In [21]:
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston!",
    }
]
response = client.chat.completions.create(
    model=llm_model,
    messages=messages,
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "get_current_weather"}},
)
print(response)

ChatCompletion(id='chatcmpl-DSaEjjtg6wSPBEYW7lFtJwcnucz9B', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_xHWcVpx82rVGjlGGOUvuyFZS', function=Function(arguments='{"location":"Boston, MA"}', name='get_current_weather'), type='function')]))], created=1775704849, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_e2564de534', usage=CompletionUsage(completion_tokens=7, prompt_tokens=89, total_tokens=96, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [22]:
# Append assistant message (with tool_calls) to conversation
messages.append(response.choices[0].message)

In [23]:
tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
observation = get_current_weather(args)

In [24]:
# "tool" role replaces "function" role, with tool_call_id reference
messages.append(
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": observation,
    }
)

In [25]:
response = client.chat.completions.create(
    model=llm_model,
    messages=messages,
)
print(response.choices[0].message.content)

The current weather in Boston, MA is 72°F, and it's sunny and windy.
